# Notebook 13 — Reusable Scientific Code

**Module:** 01 — Python & Scientific Computing  
**Notebook:** 13 of 20  
**Prerequisites:** Notebook 12  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Functions that only work in a single notebook are not scientific assets — they're
experimental scaffolding. A function in `compbio_utils` that has a docstring, type
hints, unit tests, and a version is a reusable scientific asset. The difference
is engineering discipline, not knowledge.

This notebook applies the 'Good Enough Practices' from Wilson et al. (2017)
(studied in Module 00) to the code written so far in Module 01.

---
## Step 2 — Intuition

Reusable code has four properties:
1. **Correct** — passes tests including edge cases
2. **Documented** — docstring explains what, not how
3. **Typed** — type hints make the interface explicit
4. **Importable** — lives in a package, not a notebook

The notebook is the laboratory. The package is the published result.

---
## Step 3 — Biological Background

Scientific software is held to a different standard than research code: it is read,
cited, and reused by others. The Wilson et al. (2017) '24 practices' study found that
the most impactful practices are version control, testing, and documentation — in that
order. This notebook applies all three.

**Track A relevance:** In a bioinformatics engineering role, code review is part of the
job. Demonstrating that you write review-ready code — not just working code — is a
significant differentiator.

---
## Step 4 — Mathematical Explanation

Not applicable — this notebook is about software engineering, not mathematical methods.

---
## Step 5 — Computational Explanation

**The five refactoring moves applied in this notebook:**

1. **Extract function** — move inline code into a named, testable function
2. **Add type hints** — make input/output types machine-checkable
3. **Write docstring** — NumPy format: Parameters, Returns, Raises, Examples
4. **Move to module** — add to `compbio_utils/*.py`
5. **Write tests** — at minimum: happy path + one edge case

---
## Step 6 — Python Implementation

In [ ]:
# Cell 6.1 — Before: inline, untested, untyped
import numpy as np

# Bad: no docstring, no types, no tests, inline in notebook
def fold_change(tumour, normal):
    return tumour.mean(axis=1) / normal.mean(axis=1)

In [ ]:
# Cell 6.2 — After: typed, documented, handles edge cases
def log2_fold_change(
    treatment: np.ndarray,
    control: np.ndarray,
    pseudocount: float = 1.0,
) -> np.ndarray:
    """
    Compute log2 fold change between treatment and control.

    Parameters
    ----------
    treatment : array of shape (n_genes, n_treatment_samples)
        Expression counts in the treatment group.
    control : array of shape (n_genes, n_control_samples)
        Expression counts in the control group.
    pseudocount : float, default 1.0
        Added to means before log-transformation to avoid log(0).

    Returns
    -------
    np.ndarray of shape (n_genes,)
        log2(mean_treatment + pseudocount) - log2(mean_control + pseudocount)

    Raises
    ------
    ValueError
        If treatment and control do not have the same number of rows.

    Examples
    --------
    >>> t = np.array([[8.0, 8.0], [2.0, 2.0]])
    >>> c = np.array([[4.0, 4.0], [4.0, 4.0]])
    >>> log2_fold_change(t, c, pseudocount=0)
    array([ 1., -1.])
    """
    if treatment.shape[0] != control.shape[0]:
        raise ValueError(
            f"treatment has {treatment.shape[0]} genes, "
            f"control has {control.shape[0]} — must match."
        )
    if pseudocount < 0:
        raise ValueError(f"pseudocount must be ≥ 0, got {pseudocount}")
    mean_t = treatment.mean(axis=1) + pseudocount
    mean_c = control.mean(axis=1) + pseudocount
    return np.log2(mean_t) - np.log2(mean_c)

# Quick smoke test
t = np.array([[8.0, 8.0], [2.0, 2.0]])
c = np.array([[4.0, 4.0], [4.0, 4.0]])
print(log2_fold_change(t, c, pseudocount=0))  # [1. -1.]

In [ ]:
# Cell 6.3 — Ruff for style checking
import subprocess, textwrap, pathlib, tempfile

bad_code = textwrap.dedent("""
    import numpy as np,os
    def FC(t,n):
        x=t.mean(axis=1)/n.mean(axis=1)
        return x
""")

with tempfile.NamedTemporaryFile(suffix=".py", mode="w", delete=False) as f:
    f.write(bad_code)
    tmp_path = f.name

result = subprocess.run(["ruff", "check", tmp_path], capture_output=True, text=True)
print("Ruff output:")
print(result.stdout or result.stderr or "(no issues found)")

In [ ]:
# Cell 6.4 — Module audit: is each function in compbio_utils complete?
import pathlib

repo_root = pathlib.Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / ".git").exists(): break
    repo_root = repo_root.parent

cbu_path = repo_root / "utilities" / "compbio_utils"

EXPECTED_FUNCTIONS = {
    "sequence.py": ["gc_content", "complement", "reverse_complement", "sliding_gc"],
    "stats.py":    ["cpm_normalize", "zscore_columns", "pca_svd", "bootstrap_ci",
                    "log2_fold_change", "fit_michaelis_menten"],
    "plotting.py": ["pca_scatter", "volcano_plot", "expression_heatmap"],
    "io.py":       ["read_fasta", "clean_expression_matrix"],
}

print("compbio_utils audit:")
for filename, funcs in EXPECTED_FUNCTIONS.items():
    fpath = cbu_path / filename
    content = fpath.read_text() if fpath.exists() else ""
    print(f"\n  {filename}:")
    for fn in funcs:
        present = f"def {fn}" in content
        print(f"    {'✓' if present else '✗'}  {fn}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Code quality dashboard
import matplotlib.pyplot as plt

categories = ["Type hints", "Docstrings", "Unit tests", "In package", "Ruff clean"]
# Score each: 0 = not done, 1 = done for some, 2 = done for all
scores = [2, 1, 1, 1, 1]  # ← update these as you complete the exercises

fig, ax = plt.subplots(figsize=(7, 3))
colors = ["tomato" if s == 0 else "orange" if s == 1 else "seagreen" for s in scores]
ax.barh(categories, scores, color=colors)
ax.set_xlim(0, 2)
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["Not done", "Partial", "Complete"])
ax.set_title("compbio_utils code quality — Module 01")
plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Move `log2_fold_change` into `compbio_utils/stats.py` and write 5 unit tests:
   happy path, pseudocount=0, shape mismatch, negative pseudocount, single-gene input.
2. Run `ruff check utilities/compbio_utils/` and fix all warnings.
3. For each function in `EXPECTED_FUNCTIONS`, verify it has: type hints, docstring, ≥1 test.
   Fill in missing ones before Notebook 14.
4. What is the NumPy docstring convention? How does it differ from Google style?
   Write `log2_fold_change`'s docstring in both styles.

---
## Quiz — Active Recall

1. What are the four properties of reusable scientific code?
2. What does ruff check? What does it NOT check?
3. Why does `log2_fold_change` use `pseudocount` instead of checking for zero means?
4. What is the difference between a docstring and a comment?
5. Name three functions currently missing from `compbio_utils` that will be needed
   in Module 03 (Statistics and Probability).

---
## Reflection

**Date completed:** ____________________

> *[Did the audit in Cell 6.4 find any gaps? Which functions still need tests? Is ruff clean?]*

---
*Next: `14_testing_with_pytest.ipynb`*